In [1]:
!pip install tensorboardX rdkit
!pip install rdkit-pypi
import os
from sklearn.model_selection import train_test_split
import torch
import torch.autograd as autograd
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torch.utils.data as Data
torch.manual_seed(8) # for reproduce
import numpy as np
from collections import defaultdict
from sklearn.model_selection import LeaveOneOut
import time
from tqdm import tqdm
import numpy as np
import gc
import sys
sys.path.append('/content/drive/MyDrive/Colab Notebooks/Posdoc/Native_AFP/code/')
sys.setrecursionlimit(50000)
import pickle
torch.backends.cudnn.benchmark = True
torch.set_default_tensor_type('torch.cuda.FloatTensor')
# from tensorboardX import SummaryWriter
torch.nn.Module.dump_patches = True
import copy
import pandas as pd
#then import my own modules
from GCN import Fingerprint, GCNModel, save_smiles_dicts, get_smiles_dicts, get_smiles_array, moltosvg_highlight
from rdkit import Chem
# from rdkit.Chem import AllChem
from rdkit.Chem import QED
from rdkit.Chem import rdMolDescriptors, MolSurf
from rdkit.Chem.Draw import SimilarityMaps
from rdkit import Chem
from rdkit.Chem import AllChem
from rdkit.Chem import rdDepictor
from rdkit.Chem.Draw import rdMolDraw2D
%matplotlib inline
from numpy.polynomial.polynomial import polyfit
import matplotlib.pyplot as plt
from matplotlib import gridspec
import matplotlib.cm as cm
import matplotlib
import seaborn as sns; sns.set_style("darkgrid")
from IPython.display import SVG, display

import itertools
from sklearn.metrics import r2_score
import scipy
import random

import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from sklearn.preprocessing import StandardScaler
import pickle
import os
import time
from rdkit import Chem
from sklearn.model_selection import LeaveOneOut
from collections import defaultdict


device= torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 101.7/101.7 kB 2.3 MB/s eta 0:00:00 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 34.3/34.3 MB 50.6 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 29.4/29.4 MB 55.1 MB/s eta 0:00:00:00:0100:01


/usr/local/lib/python3.11/dist-packages/torch/__init__.py:614: UserWarning: torch.set_default_tensor_type() is deprecated as of PyTorch 2.1, please use torch.set_default_dtype() and torch.set_default_device() as alternatives. (Triggered internally at ../torch/csrc/tensor/python_tensor.cpp:451.)
  _C._set_default_tensor_type(t)


cuda


In [2]:
def create_structured_absolute_ecfp_dictionary(df ,split_calixarene_dict, holdout_size=0.5):
    """
    A complementary function to that above, but for absolute training/prediction rather than relative.
    Returns train and test dataframes with targets as columns.
    """
    # Get the processed df
    calixarene_df = df

    # Always looking at all columns
    target_columns = ['H3K4', 'H3K4ac', 'H3K4me1', 'H3K4me2', 'H3K4me3', 'H3K9me3', 'H3R2me2a', 'H3R2me2s','cano_smiles']

    # Determine the holdout calixarenes - int() always rounds down
    holdout_pred_amount = int(len(split_calixarene_dict['predictable']) * holdout_size)
    holdout_unpred_amount = int(len(split_calixarene_dict['unpredictable']) * holdout_size)

    holdout_calixarenes_pred = random.sample(split_calixarene_dict['predictable'], holdout_pred_amount)
    holdout_calixarenes_unpred = random.sample(split_calixarene_dict['unpredictable'], holdout_unpred_amount)
    all_holdout_calix = holdout_calixarenes_pred + holdout_calixarenes_unpred
    
    # Create empty lists to store data for train and test dataframes
    train_data = []
    test_data = []
    
    # Group the dataframe by 'Host' to process each host once
    grouped_df = calixarene_df.groupby('Host')
    
    for host, group in grouped_df:
        # Get the first row for this host (assuming SMILES is the same for all rows with the same host)
        row = group.iloc[0]
        
        # Create a base dictionary with host and SMILES
        base_record = {
            'Host': host,
            'SMILES': row['SMILES']
        }
        
        # Add target values
        for column in target_columns:
            base_record[column] = row[column]
        
        # Add to appropriate dataset
        if host not in all_holdout_calix:
            train_data.append(base_record)
        elif host in all_holdout_calix:
            test_data.append(base_record)
    
    # Convert lists to dataframes
    train_df = pd.DataFrame(train_data)
    test_df = pd.DataFrame(test_data)
    
    return train_df, test_df

split_calix_dict = {'predictable': ['AP1', 'AP3', 'AP4', 'AP5', 'AP6',
                                    'AP7', 'AP8', 'AP9', 'AH1', 'AH2',
                                    'AH3', 'AH4', 'AH5', 'AH6', 'AH7',
                                    'AM1', 'AM2', 'AO1', 'AO2', 'AO3',
                                    'E1', 'E3', 'E6', 'E7', 'E8', 'E11',
                                    'P-NO2', 'PSC4'],
                    'unpredictable': ['BP0', 'BP1', 'BH2', 'BM1', 'CP1',
                                      'CP2', 'DP2', 'DM1', 'DO2', 'DO3']}




In [3]:
def prepare_model_and_data(raw_filename, target_name='Calx', targets=None, batch_size=50, epochs=800, p_dropout=0.1, fingerprint_dim=150, weight_decay=2, learning_rate=3, radius=3, T=2, per_target_output_units_num=1):
    if targets is None:
        targets = ['H3K4', 'H3K4ac', 'H3K4me1', 'H3K4me2', 'H3K4me3', 'H3K9me3', 'H3R2me2a', 'H3R2me2s']

    feature_filename = raw_filename.replace('.csv', '.pickle')
    filename = raw_filename.replace('.csv', '')
    prefix_filename = raw_filename.split('/')[-1].replace('.csv', '')
    smiles_targets_df = pd.read_csv(raw_filename)

    smilesList = smiles_targets_df.SMILES.values
    print("number of all smiles: ", len(smilesList))

    atom_num_dist = []
    remained_smiles = []
    canonical_smiles_list = []
    for smiles in smilesList:
        try:
            mol = Chem.MolFromSmiles(smiles)
            atom_num_dist.append(len(mol.GetAtoms()))
            remained_smiles.append(smiles)
            canonical_smiles_list.append(Chem.MolToSmiles(Chem.MolFromSmiles(smiles), isomericSmiles=True))
        except:
            print(smiles)
            pass
    print("number of successfully processed smiles: ", len(remained_smiles))

    smiles_targets_df = smiles_targets_df[smiles_targets_df["SMILES"].isin(remained_smiles)]
    smiles_targets_df['cano_smiles'] = canonical_smiles_list

    start_time = str(time.ctime()).replace(':', '-').replace(' ', '_')
    output_units_num = len(targets) * per_target_output_units_num

    if os.path.isfile(feature_filename):
        feature_dicts = pickle.load(open(feature_filename, "rb"))
    else:
        feature_dicts = save_smiles_dicts(smilesList, filename)
    feature_dicts = get_smiles_dicts(smilesList)

    remained_df = smiles_targets_df[smiles_targets_df["cano_smiles"].isin(feature_dicts['smiles_to_atom_mask'].keys())]

    x_atom, x_bonds, x_atom_index, x_bond_index, x_mask, smiles_to_rdkit_list = get_smiles_array([canonical_smiles_list[0]], feature_dicts)
    num_atom_features = x_atom.shape[-1]
    num_bond_features = x_bonds.shape[-1]

    loss_function = nn.MSELoss()
    model = Fingerprint(radius, T, num_atom_features, num_bond_features, fingerprint_dim, output_units_num, p_dropout)
    model.to(device)

    optimizer = optim.Adam(model.parameters(), 10**-learning_rate, weight_decay=10**-weight_decay)

    model_parameters = filter(lambda p: p.requires_grad, model.parameters())
    params = sum([np.prod(p.size()) for p in model_parameters])
    #print(params)
    #for name, param in model.named_parameters():
     #   if param.requires_grad:
      #      print(name, param.data.shape)


    return model, optimizer, loss_function, remained_df, targets, feature_dicts



In [4]:
def train(model, dataset, optimizer, loss_function, epoch, batch_size, targets, feature_dicts, ratio_list):
    model.train()
    
    if len(dataset) <= batch_size:
        batch_list = [dataset.index]

    else:
        valList = np.arange(0, dataset.shape[0])
        np.random.shuffle(valList)
        batch_list = [valList[i:i+batch_size] for i in range(0, dataset.shape[0], batch_size)]
    
    device = next(model.parameters()).device
    total_loss = 0
    
    for counter, batch in enumerate(batch_list):
        batch_df = dataset.loc[batch, :]
        smiles_list = batch_df.cano_smiles.values

        x_atom, x_bonds, x_atom_index, x_bond_index, x_mask, _ = get_smiles_array(smiles_list, feature_dicts)
        
        x_atom = torch.Tensor(x_atom).to(device)
        #print(x_atom.shape)
        x_bonds = torch.Tensor(x_bonds).to(device)
        x_atom_index = torch.LongTensor(x_atom_index).to(device)
        x_bond_index = torch.LongTensor(x_bond_index).to(device)
        x_mask = torch.Tensor(x_mask).to(device)
        
        atoms_prediction, mol_prediction = model(x_atom, x_bonds, x_atom_index, x_bond_index, x_mask)
        
        optimizer.zero_grad()
        loss = 0.0
        
        for i, target in enumerate(targets):
            y_pred = mol_prediction[:, i]
            y_val = torch.Tensor(batch_df[target].values).squeeze().to(device)
            target_loss = loss_function(y_pred, y_val) * (ratio_list[i] ** 2)
            loss += target_loss
        
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
    
    return total_loss / len(batch_list)

def eval(model, dataset, targets, feature_dicts, batch_size):
    model.eval()
    
    eval_MAE_list = {target: [] for target in targets}
    eval_MSE_list = {target: [] for target in targets}
    y_val_list = {target: [] for target in targets}
    y_pred_list = {target: [] for target in targets}
    smiles_list = []
    
    if len(dataset) <= batch_size:
        batch_list = [dataset.index]
    else:
        valList = np.arange(0, dataset.shape[0])
        batch_list = [valList[i:i+batch_size] for i in range(0, dataset.shape[0], batch_size)]
    
    device = next(model.parameters()).device
    
    with torch.no_grad():
        for batch in batch_list:
            batch_df = dataset.loc[batch, :]
            batch_smiles = batch_df.cano_smiles.values
            smiles_list.extend(batch_smiles)
            
            x_atom, x_bonds, x_atom_index, x_bond_index, x_mask, _ = get_smiles_array(batch_smiles, feature_dicts)
            
            x_atom = torch.Tensor(x_atom).to(device)
            x_bonds = torch.Tensor(x_bonds).to(device)
            x_atom_index = torch.LongTensor(x_atom_index).to(device)
            x_bond_index = torch.LongTensor(x_bond_index).to(device)
            x_mask = torch.Tensor(x_mask).to(device)
            
            atoms_prediction, mol_prediction = model(x_atom, x_bonds, x_atom_index, x_bond_index, x_mask)
            
            for i, target in enumerate(targets):
                y_pred = mol_prediction[:, i].view(-1, 1)
                y_val = torch.Tensor(batch_df[target].values.reshape(-1, 1)).to(device)
                
                MAE = F.l1_loss(y_pred, y_val, reduction='none')
                MSE = F.mse_loss(y_pred, y_val, reduction='none')
                
                y_pred_list[target].extend(y_pred.cpu().numpy().flatten())
                y_val_list[target].extend(y_val.cpu().numpy().flatten())
                
                eval_MAE_list[target].extend(MAE.cpu().numpy().flatten())
                eval_MSE_list[target].extend(MSE.cpu().numpy().flatten())
    
    eval_MAE = np.array([np.mean(eval_MAE_list[target]) for target in targets])
    eval_MSE = np.array([np.mean(eval_MSE_list[target]) for target in targets])
    
    return eval_MAE, eval_MSE, y_pred_list, y_val_list, smiles_list

def train_and_evaluate(model, train_df, test_df, targets, feature_dicts, optimizer, loss_function, 
                      batch_size, num_epochs, patience=30, min_delta=0.001, 
                      prefix_filename='', start_time=''):
    # Initialize tracking structures
    best_param = {
        "train_epoch": 0, 
        "val_epoch": 0, 
        "train_MSE": float('inf'), 
        "val_MSE": float('inf')
    }
    
    # Prediction containers
    prediction_containers = {
        'train': {t: {'pred': [], 'actual': []} for t in targets},
        'test': {t: {'pred': [], 'actual': []} for t in targets}
    }
    
    # Create validation set from training data
    # Create validation set from training data
    val_size = max(5, int(0.1 * len(train_df)))
    
    # Try to stratify, but handle the case where it's not possible
    try:
        # Attempt to create fewer bins for small datasets
        n_bins = min(5, max(2, len(train_df) // 3))
        train_subset, val_df = train_test_split(
            train_df, 
            test_size=val_size,
            stratify=pd.qcut(train_df[targets[0]], q=n_bins, duplicates='drop') if len(targets) > 0 else None
        )
    except ValueError:
        # Fall back to random split if stratification fails
        print("Warning: Stratification failed, using random split instead")
        train_subset, val_df = train_test_split(
            train_df, 
            test_size=val_size
        )
    
    
    # --- Training Loop ---
    best_val_mse = float('inf')
    epochs_no_improve = 0
    best_model_state = None
    best_val_metrics = None
    
    for epoch in range(num_epochs):
        # Training phase
        train_loss = train(model, train_subset, optimizer, loss_function, epoch, 
                          batch_size, targets, feature_dicts, ratio_list=[1.0]*len(targets))
        
        # Validation evaluation
        val_metrics = eval(model, val_df, targets, feature_dicts, batch_size)
        val_mae, val_mse = val_metrics[0].mean(), val_metrics[1].mean()
        
        # Model checkpointing based on validation
        if val_mse < best_val_mse - min_delta:
            best_val_mse = val_mse
            epochs_no_improve = 0
            best_model_state = model.state_dict()
            best_val_metrics = val_metrics
            best_param.update({
                "val_epoch": epoch,
                "val_MSE": val_mse,
                "train_epoch": epoch,
                "train_MSE": train_loss  # Using actual training loss
            })
        else:
            epochs_no_improve += 1
        
        if epochs_no_improve >= patience:
            print(f"Early stopping at epoch {epoch+1}")
            break
    
    # --- Post-Training Evaluation ---
    # Load best model
    model.load_state_dict(best_model_state)
    
    # Re-evaluate all sets with best model
    final_train_metrics = eval(model, train_subset, targets, feature_dicts, batch_size)
    final_val_metrics = best_val_metrics
    test_metrics = eval(model, test_df, targets, feature_dicts, batch_size)
    test_MAE, test_MSE, test_pred, test_actual, _ = test_metrics
    
    # Store host-level results
    host_results = {}
    for i, host in enumerate(test_df['Host'].values):
        host_results[host] = {
            target: {
                "actual": float(test_actual[target][i]),  # Convert to Python float
                "predicted": float(test_pred[target][i])
            }
            for target in targets
        }
    
    # Store predictions
    for target in targets:
        # Training data (from final evaluation)
        prediction_containers['train'][target]['pred'].extend(final_train_metrics[2][target])
        prediction_containers['train'][target]['actual'].extend(final_train_metrics[3][target])
        
        # Test data
        prediction_containers['test'][target]['pred'].extend(test_metrics[2][target])
        prediction_containers['test'][target]['actual'].extend(test_metrics[3][target])
    
    # --- Results Summary ---
    print("\nTraining Summary:")
    print(f"Train MSE: {np.mean(final_train_metrics[1]):.4f}")
    print(f"Val MSE: {np.mean(final_val_metrics[1]):.4f}")
    print(f"Test MSE: {np.mean(test_metrics[1]):.4f}")
    
    overall_train_mae = np.mean(final_train_metrics[0])
    overall_train_mse = np.mean(final_train_metrics[1])
    overall_test_mae = np.mean(test_metrics[0])
    overall_test_mse = np.mean(test_metrics[1])
    
    print("\nFinal Performance:")
    print(f"Train MAE: {overall_train_mae:.4f}")
    print(f"Test MAE: {overall_test_mae:.4f}")
    print(f"Train MSE: {overall_train_mse:.4f}")
    print(f"Test MSE: {overall_test_mse:.4f}")
    
    # Create a single fold result for compatibility with previous code
    fold_result = {
        'fold': 1,
        'train_metrics': {
            'MAE': final_train_metrics[0].tolist(),
            'MSE': final_train_metrics[1].tolist()
        },
        'val_metrics': {
            'MAE': final_val_metrics[0].tolist(),
            'MSE': final_val_metrics[1].tolist()
        },
        'test_metrics': {
            'MAE': test_metrics[0].tolist(),
            'MSE': test_metrics[1].tolist()
        }
    }
    
    return {
        "host_predictions": host_results,  
        "overall_metrics": (overall_train_mae, overall_train_mse, 
                           overall_test_mae, overall_test_mse),
        "fold_results": [fold_result],  # Wrap in list for compatibility
        "train_predictions": prediction_containers['train'],
        "test_predictions": prediction_containers['test']
    }


In [5]:
''''raw_filename= '/notebooks/Codebase/Database/cal_abs.csv'

# Prepare the model 
model, optimizer, loss_function, remained_df, targets, feature_dicts= prepare_model_and_data(raw_filename,
                                                                                            target_name='Calx', targets=None, batch_size=38, epochs=800,
                                                                                             p_dropout=0.1, fingerprint_dim=150, weight_decay=2, learning_rate=3, radius=3,
                                                                                             T=2, per_target_output_units_num=1)


# Prepare the test and train dataframes based on the code Jeff had

train_df, test_df = create_structured_absolute_ecfp_dictionary(remained_df, split_calixarene_dict=split_calix_dict, holdout_size=0.1)


#train and get the Results. 
results = train_and_evaluate(model, train_df, test_df, targets, feature_dicts, optimizer, loss_function, 
                       batch_size=38 , num_epochs =800,  patience=30, min_delta=0.001)'''                                      

"'raw_filename= '/notebooks/Codebase/Database/cal_abs.csv'\n\n# Prepare the model \nmodel, optimizer, loss_function, remained_df, targets, feature_dicts= prepare_model_and_data(raw_filename,\n                                                                                            target_name='Calx', targets=None, batch_size=38, epochs=800,\n                                                                                             p_dropout=0.1, fingerprint_dim=150, weight_decay=2, learning_rate=3, radius=3,\n                                                                                             T=2, per_target_output_units_num=1)\n\n\n# Prepare the test and train dataframes based on the code Jeff had\n\ntrain_df, test_df = create_structured_absolute_ecfp_dictionary(remained_df, split_calixarene_dict=split_calix_dict, holdout_size=0.1)\n\n\n#train and get the Results. \nresults = train_and_evaluate(model, train_df, test_df, targets, feature_dicts, optimizer, loss_function, \n 

In [6]:
def run_multiple_iterations(raw_filename, split_calix_dict, holdout_size=0.1, num_iterations=20):
    """
    Runs the model preparation, training, and evaluation multiple times and collects results.
    
    Args:
        raw_filename: Path to the raw data file
        split_calix_dict: Dictionary with 'predictable' and 'unpredictable' host lists
        holdout_size: Fraction of data to use for testing
        num_iterations: Number of iterations to run
        
    Returns:
        Dictionary with results structured as {iteration: {host: [(pred1, actual1), (pred2, actual2), ...]}}
    """
    # Target columns in order
    target_columns = ['H3K4', 'H3K4ac', 'H3K4me1', 'H3K4me2', 'H3K4me3', 'H3K9me3', 'H3R2me2a', 'H3R2me2s']
    
    # Dictionary to store final results
    final_results = {}
    
    # List of predictable hosts for filtering
    predictable_hosts = split_calix_dict['predictable']
    
    for i in range(num_iterations):
        print(f"Running iteration {i+1}/{num_iterations}")
        
        # 1. Prepare the model and data
        model, optimizer, loss_function, remained_df, targets, feature_dicts = prepare_model_and_data(
            raw_filename,
            target_name='Calx', 
            targets=None, 
            batch_size=38, 
            epochs=800,
            p_dropout=0.1, 
            fingerprint_dim=150, 
            weight_decay=2, 
            learning_rate=3, 
            radius=3,
            T=2, 
            per_target_output_units_num=1
        )
        
        # 2. Create train and test dataframes
        train_df, test_df = create_structured_absolute_ecfp_dictionary(
            remained_df, 
            split_calixarene_dict=split_calix_dict, 
            holdout_size=holdout_size
        )
        
        # 3. Train and evaluate the model
        results = train_and_evaluate(
            model, 
            train_df, 
            test_df, 
            targets, 
            feature_dicts, 
            optimizer, 
            loss_function, 
            batch_size=38, 
            num_epochs=800, 
            patience=30, 
            min_delta=0.001
        )
        
        # 4. Extract and format results for this iteration
        iteration_dict = {}
        
        # Get host predictions from results
        host_predictions = results["host_predictions"]
        
        # Filter to include only predictable hosts
        for host, predictions in host_predictions.items():
            if host in predictable_hosts:
                # Create list of (predicted, actual) tuples for each target
                host_results = []
                
                for target in target_columns:
                    if target in predictions:
                        pred_value = predictions[target]["predicted"]
                        actual_value = predictions[target]["actual"]
                        host_results.append((pred_value, actual_value))
                
                # Add to iteration dictionary
                iteration_dict[host] = host_results
        
        # Add iteration results to final dictionary
        final_results[str(i)] = iteration_dict
    
    return final_results

# Usage example:
raw_filename = '/notebooks/Codebase/Database/cal_abs.csv'
split_calix_dict = {
    'predictable': ['AP1', 'AP3', 'AP4', 'AP5', 'AP6', 'AP7', 'AP8', 'AP9', 'AH1', 'AH2', 
                    'AH3', 'AH4', 'AH5', 'AH6', 'AH7', 'AM1', 'AM2', 'AO1', 'AO2', 'AO3', 
                    'E1', 'E3', 'E6', 'E7', 'E8', 'E11', 'P-NO2', 'PSC4'],
    'unpredictable': ['BP0', 'BP1', 'BH2', 'BM1', 'CP1', 'CP2', 'DP2', 'DM1', 'DO2', 'DO3']
}

# Run 20 iterations and collect results

holdout_size= [0.05,0.1,0.15,0.25,0.5,0.75]

for i in holdout_size:
    all_results = run_multiple_iterations(raw_filename, split_calix_dict, holdout_size=i, num_iterations=20)
    output = open(f'/notebooks/Codebase/Result_dict/20 split {i} AttentiveFP abso_val.pkl', 'wb')
    pickle.dump(all_results, output)
    output.close()

Running iteration 1/20
number of all smiles:  38
number of successfully processed smiles:  38
Early stopping at epoch 32

Training Summary:
Train MSE: 2.9003
Val MSE: 3.7514
Test MSE: 0.2998

Final Performance:
Train MAE: 1.3139
Test MAE: 0.4308
Train MSE: 2.9003
Test MSE: 0.2998
Running iteration 2/20
number of all smiles:  38
number of successfully processed smiles:  38
Early stopping at epoch 105

Training Summary:
Train MSE: 1.4913
Val MSE: 1.5129
Test MSE: 1.1723

Final Performance:
Train MAE: 0.8580
Test MAE: 1.0106
Train MSE: 1.4913
Test MSE: 1.1723
Running iteration 3/20
number of all smiles:  38
number of successfully processed smiles:  38
Early stopping at epoch 33

Training Summary:
Train MSE: 2.1245
Val MSE: 6.9958
Test MSE: 1.6449

Final Performance:
Train MAE: 1.1736
Test MAE: 1.2253
Train MSE: 2.1245
Test MSE: 1.6449
Running iteration 4/20
number of all smiles:  38
number of successfully processed smiles:  38
Early stopping at epoch 64

Training Summary:
Train MSE: 2.835

In [ ]:
#_______________Reference_code_from_Jeff_Repo__________________________

'''
def create_structured_relative_ecfp_dictionary(calixarene_csv_folder,
                                               calixarene_csv_file,
                                               split_calixarene_dict,
                                               holdout_size):
    """
    A complementary function to that directly above, just for relative training/prediction rather than absolute

    Only method being used for fingerprints is 'concat', based on previous testing

    To this dict, add 'test calix' as a key, with the list of selected calixarenes for testing. Unlike in absolute training,
    it's not immediately obvious which calix is being tested.
    """

    # Read in the .csv file
    calixarene_df = pd.read_csv(calixarene_csv_folder + calixarene_csv_file)

    calixarene_comparison_dict = {}
    calixarene_comparison_dict['train'] = {}
    calixarene_comparison_dict['test'] = {}

    # Always looking at all columns
    target_columns = ['H3K4', 'H3K4ac', 'H3K4me1', 'H3K4me2', 'H3K4me3', 'H3K9me3', 'H3R2me2a', 'H3R2me2s']

    # Determine the holdout calixarenes - int() always rounds down
    holdout_pred_amount = int(len(split_calixarene_dict['predictable']) * holdout_size)
    holdout_unpred_amount = int(len(split_calixarene_dict['unpredictable']) * holdout_size)

    holdout_calixarenes_pred = random.sample(split_calixarene_dict['predictable'], holdout_pred_amount)
    holdout_calixarenes_unpred = random.sample(split_calixarene_dict['unpredictable'], holdout_unpred_amount)
    all_holdout_calix = holdout_calixarenes_pred + holdout_calixarenes_unpred
    calixarene_comparison_dict['holdout'] = all_holdout_calix

    # Iterate over all combinations of two different hosts
    for (idx1, row1), (idx2, row2) in itertools.permutations(calixarene_df.iterrows(), 2):
        host_pair = (row1['Host'], row2['Host'])
        
        if row1['Host'] not in all_holdout_calix and row2['Host'] not in all_holdout_calix:
            for target in target_columns:
                key = host_pair + (target,)
                calixarene_comparison_dict['train'][key] = {'SMILES': (row1['SMILES'], row2['SMILES']),
                                                         'ECFP': CSF.create_double_ecpf6_fingerprint((row1['SMILES'], row2['SMILES']),
                                                                                                     method='concat'),
                                                         'Target_Val': row1[target] - row2[target],
                                                         'Target': target}
        else:
            if (row1['Host'] in holdout_calixarenes_pred) ^ (row2['Host'] in holdout_calixarenes_pred):
                if row1['Host'] not in holdout_calixarenes_unpred and row2['Host'] not in holdout_calixarenes_unpred:
                    for target in target_columns:
                        key = host_pair + (target,)
                        calixarene_comparison_dict['test'][key] = {'SMILES': (row1['SMILES'], row2['SMILES']),
                                                                'ECFP': CSF.create_double_ecpf6_fingerprint((row1['SMILES'], row2['SMILES']),
                                                                                                            method='concat'),
                                                                'Target_Val': row1[target] - row2[target],
                                                                'Target': target}
                        if row1['Host'] in holdout_calixarenes_pred:
                            calixarene_comparison_dict['test'][key]['test_pos'] = 'row1'
                            calixarene_comparison_dict['test'][key]['known_val'] = row2[target]
                        elif row2['Host'] in holdout_calixarenes_pred:
                            calixarene_comparison_dict['test'][key]['test_pos'] = 'row2'
                            calixarene_comparison_dict['test'][key]['known_val'] = row1[target]

    return calixarene_comparison_dict

def create_structured_ecfp_dictionary(calixarene_csv_folder,
                                      calixarene_csv_file,
                                      split_calixarene_dict,
                                      holdout_size):
    """
    A function used to test different sizes of data holdout - will there be a difference between
    absolute training and relative training?

    For the input calixarene lists, they are pre-segregated into the types of calixarenes that have been observed
    to be 'predictable' (mono and unsubstituted) and 'unpredictable' (multiple substituents)

    We only care about predictions for 'predictable' systems, so we exclude the 'unpredictable' systems from the test set.
    """

    # Read in the .csv file
    calixarene_df = pd.read_csv(calixarene_csv_folder + calixarene_csv_file)

    # Create a dictionary to hold the calixarene data
    calixarene_dict = {}
    calixarene_dict['train'] = {}
    calixarene_dict['test'] = {}

    # Determine the holdout calixarenes
    holdout_calixarenes_pred = random.sample(list(split_calixarene_dict['predictable']), holdout_size)
    holdout_calixarenes_unpred = random.sample(list(split_calixarene_dict['unpredictable']), holdout_size)

    # Always looking at all columns
    target_columns = ['H3K4', 'H3K4ac', 'H3K4me1', 'H3K4me2', 'H3K4me3', 'H3K9me3', 'H3R2me2a', 'H3R2me2s']
    # Iterate through the rows of the dataframe
    for index, row in calixarene_df.iterrows():
        #Check for duplicates and whether the host is the holdout
        if row['Host'] not in holdout_calixarenes_pred and row['Host'] not in holdout_calixarenes_unpred:
            if row['Host'] not in calixarene_dict['train']:
                for targ_no, specific_column in enumerate(target_columns):
                    calixarene_dict['train'][row['Host'] + str('_') + str(targ_no)] = {'SMILES': row['SMILES'],
                                                    'ECFP': CSF.create_ecfp6_fingerprint(row['SMILES']),
                                                    'Target_Val': row[specific_column],
                                                    'Target': specific_column}
        else:
            if row['Host'] not in calixarene_dict['test'] and row['Host'] in holdout_calixarenes_pred:
                for targ_no, specific_column in enumerate(target_columns):
                    calixarene_dict['test'][row['Host'] + str('_') + str(targ_no)] = {'SMILES': row['SMILES'],
                                                    'ECFP': CSF.create_ecfp6_fingerprint(row['SMILES']),
                                                    'Target_Val': row[specific_column],
                                                    'Target': specific_column}
    
    return calixarene_dict

def create_structured_absolute_ecfp_dictionary(split_calixarene_dict,
                                               holdout_size=0.5):
    """
    A complementary function to that above, but for absolute training/prediction rather than relative

    Add held-out calixarene names to the dictionary for later use
    """

    # Read in the .csv file
    calixarene_df = pd.read_csv('/notebooks/data/Data excluding non-binders.csv')

    calixarene_dict = {}
    calixarene_dict['train'] = {}
    calixarene_dict['test'] = {}

    # Always looking at all columns
    target_columns = ['H3K4', 'H3K4ac', 'H3K4me1', 'H3K4me2', 'H3K4me3', 'H3K9me3', 'H3R2me2a', 'H3R2me2s']

    # Determine the holdout calixarenes - int() always rounds down
    holdout_pred_amount = int(len(split_calixarene_dict['predictable']) * holdout_size)
    holdout_unpred_amount = int(len(split_calixarene_dict['unpredictable']) * holdout_size)

    holdout_calixarenes_pred = random.sample(split_calixarene_dict['predictable'], holdout_pred_amount)
    holdout_calixarenes_unpred = random.sample(split_calixarene_dict['unpredictable'], holdout_unpred_amount)
    all_holdout_calix = holdout_calixarenes_pred + holdout_calixarenes_unpred
    # Iterate through the rows of the dataframe
    for index, row in calixarene_df.iterrows():
        #Check for duplicates and whether the host is the holdout
        if row['Host'] not in all_holdout_calix:
            if row['Host'] not in calixarene_dict['train']:
                for targ_no, specific_column in enumerate(target_columns):
                    calixarene_dict['train'][row['Host'] + str('_') + str(targ_no)] = {'SMILES': row['SMILES'],
                                                    'Target_Val': row[specific_column],
                                                    'Target': specific_column}
        else:
            if row['Host'] not in calixarene_dict['test']:
                for targ_no, specific_column in enumerate(target_columns):
                    calixarene_dict['test'][row['Host'] + str('_') + str(targ_no)] = {'SMILES': row['SMILES'],
                                                    'Target_Val': row[specific_column],
                                                    'Target': specific_column}
    
    return calixarene_dict

split_calix_dict = {'predictable': ['AP1', 'AP3', 'AP4', 'AP5', 'AP6',
                                    'AP7', 'AP8', 'AP9', 'AH1', 'AH2',
                                    'AH3', 'AH4', 'AH5', 'AH6', 'AH7',
                                    'AM1', 'AM2', 'AO1', 'AO2', 'AO3',
                                    'E1', 'E3', 'E6', 'E7', 'E8', 'E11',
                                    'P-NO2', 'PSC4'],
                    'unpredictable': ['BP0', 'BP1', 'BH2', 'BM1', 'CP1',
                                      'CP2', 'DP2', 'DM1', 'DO2', 'DO3']}
create_structured_absolute_ecfp_dictionary(split_calixarene_dict=split_calix_dict)
'''


In [9]:
def create_structured_relative_ecfp_dataframes(holdout_size):
    """
    A function to create structured relative ECFP dataframes for training and testing.

    This function processes calixarene data to generate two DataFrames:
    - One for training data where neither host is in the holdout set.
    - One for testing data where one host is in the holdout set for prediction.

    Parameters:
    - holdout_size: Fraction of calixarenes to be used for testing.

    Returns:
    - Tuple of two DataFrames (train_df, test_df)
    """
    split_calixarene_dict = {'predictable': ['AP1', 'AP3', 'AP4', 'AP5', 'AP6',
                                    'AP7', 'AP8', 'AP9', 'AH1', 'AH2',
                                    'AH3', 'AH4', 'AH5', 'AH6', 'AH7',
                                    'AM1', 'AM2', 'AO1', 'AO2', 'AO3',
                                    'E1', 'E3', 'E6', 'E7', 'E8', 'E11',
                                    'P-NO2', 'PSC4'],
                    'unpredictable': ['BP0', 'BP1', 'BH2', 'BM1', 'CP1',
                                      'CP2', 'DP2', 'DM1', 'DO2', 'DO3']}
    
    # Read in the .csv file
    calixarene_df = pd.read_csv('/notebooks/Codebase/Database/cal_abs.csv')

    # Determine the holdout calixarenes
    holdout_pred_amount = int(len(split_calixarene_dict['predictable']) * holdout_size)
    holdout_unpred_amount = int(len(split_calixarene_dict['unpredictable']) * holdout_size)

    holdout_calixarenes_pred = random.sample(split_calixarene_dict['predictable'], holdout_pred_amount)
    holdout_calixarenes_unpred = random.sample(split_calixarene_dict['unpredictable'], holdout_unpred_amount)
    all_holdout_calix = holdout_calixarenes_pred + holdout_calixarenes_unpred
    
    # Print the selected test hosts that will appear in the final dataframe
    #print("Selected test hosts (predictable):", holdout_calixarenes_pred)
    #print("Selected test hosts (unpredictable):", holdout_calixarenes_unpred)

    # Initialize lists to hold data for train and test DataFrames
    train_data = []
    test_data = []

    # Target columns for comparison
    target_columns = ['H3K4', 'H3K4ac', 'H3K4me1', 'H3K4me2', 'H3K4me3', 'H3K9me3', 'H3R2me2a', 'H3R2me2s']

    # Iterate over all combinations of two different hosts
    for (idx1, row1), (idx2, row2) in itertools.permutations(calixarene_df.iterrows(), 2):
        host_pair = (row1['Host'], row2['Host'])
        
        if row1['Host'] not in all_holdout_calix and row2['Host'] not in all_holdout_calix:
            # For training data
            train_row = {'Host_Pair': host_pair, 'SMILES': (row1['SMILES'], row2['SMILES'])}
            for target in target_columns:
                train_row[target] = row1[target] - row2[target]
            train_data.append(train_row)
        else:
            if (row1['Host'] in holdout_calixarenes_pred) ^ (row2['Host'] in holdout_calixarenes_pred):
                if row1['Host'] not in holdout_calixarenes_unpred and row2['Host'] not in holdout_calixarenes_unpred:
                    # For test data
                    test_row = {'Host_Pair': host_pair, 'SMILES': (row1['SMILES'], row2['SMILES'])}
                    for target in target_columns:
                        test_row[target] = row1[target] - row2[target]
                    # Identify which host is the test host (for information only)
                    test_host = row1['Host'] if row1['Host'] in holdout_calixarenes_pred else row2['Host']
                    #print(f"Adding test pair with test host: {test_host}, pair: {host_pair}")
                    test_data.append(test_row)

    # Convert lists to DataFrames
    train_df = pd.DataFrame(train_data)
    test_df = pd.DataFrame(test_data)

    return train_df, test_df


In [11]:
_,test_df=create_structured_relative_ecfp_dataframes(holdout_size=0.5)

Selected test hosts (predictable): ['AO3', 'AP1', 'AP7', 'AM1', 'E11', 'AO2', 'AH2', 'AP3', 'AH4', 'AO1', 'AP8', 'P-NO2', 'AP5', 'AM2']
Selected test hosts (unpredictable): ['BP0', 'BP1', 'BM1', 'DP2', 'DO3']
Adding test pair with test host: AP8, pair: ('AP8', 'AH3')
Adding test pair with test host: AP8, pair: ('AP8', 'AH1')
Adding test pair with test host: AP8, pair: ('AP8', 'DM1')
Adding test pair with test host: AP8, pair: ('AP8', 'AP9')
Adding test pair with test host: AP8, pair: ('AP8', 'AP6')
Adding test pair with test host: AP8, pair: ('AP8', 'E3')
Adding test pair with test host: AP8, pair: ('AP8', 'E6')
Adding test pair with test host: AP8, pair: ('AP8', 'CP2')
Adding test pair with test host: AP8, pair: ('AP8', 'E7')
Adding test pair with test host: AP8, pair: ('AP8', 'BH2')
Adding test pair with test host: AP8, pair: ('AP8', 'PSC4')
Adding test pair with test host: AP8, pair: ('AP8', 'AH7')
Adding test pair with test host: AP8, pair: ('AP8', 'E8')
Adding test pair with test 

In [12]:
for i in test_df['Host_Pair']:
    print (i)

('AP8', 'AH3')
('AP8', 'AH1')
('AP8', 'DM1')
('AP8', 'AP9')
('AP8', 'AP6')
('AP8', 'E3')
('AP8', 'E6')
('AP8', 'CP2')
('AP8', 'E7')
('AP8', 'BH2')
('AP8', 'PSC4')
('AP8', 'AH7')
('AP8', 'E8')
('AP8', 'AH6')
('AP8', 'AH5')
('AP8', 'AP4')
('AP8', 'E1')
('AP8', 'DO2')
('AP8', 'CP1')
('AH4', 'AH3')
('AH4', 'AH1')
('AH4', 'DM1')
('AH4', 'AP9')
('AH4', 'AP6')
('AH4', 'E3')
('AH4', 'E6')
('AH4', 'CP2')
('AH4', 'E7')
('AH4', 'BH2')
('AH4', 'PSC4')
('AH4', 'AH7')
('AH4', 'E8')
('AH4', 'AH6')
('AH4', 'AH5')
('AH4', 'AP4')
('AH4', 'E1')
('AH4', 'DO2')
('AH4', 'CP1')
('AH3', 'AP8')
('AH3', 'AH4')
('AH3', 'AP7')
('AH3', 'E11')
('AH3', 'P-NO2')
('AH3', 'AO1')
('AH3', 'AP1')
('AH3', 'AP3')
('AH3', 'AP5')
('AH3', 'AO2')
('AH3', 'AM2')
('AH3', 'AH2')
('AH3', 'AO3')
('AH3', 'AM1')
('AH1', 'AP8')
('AH1', 'AH4')
('AH1', 'AP7')
('AH1', 'E11')
('AH1', 'P-NO2')
('AH1', 'AO1')
('AH1', 'AP1')
('AH1', 'AP3')
('AH1', 'AP5')
('AH1', 'AO2')
('AH1', 'AM2')
('AH1', 'AH2')
('AH1', 'AO3')
('AH1', 'AM1')
('DM1', 'AP8')

In [15]:
test_df['SMILES'][0]

('OC1=C(CC2=CC(S(=O)(O)=O)=CC(CC3=CC(S(=O)(O)=O)=CC4=C3O)=C2O)C=C(C5=CC=C(C(N[C@@H](CC6=CC=C(O)C=C6)C(O)=O)=O)C=C5)C=C1CC7=C(O)C(C4)=CC(S(=O)(O)=O)=C7',
 'OC1=C(CC2=CC(S(=O)(O)=O)=CC(CC3=CC(S(=O)(O)=O)=CC4=C3O)=C2O)C=C(C5=CC=CO5)C=C1CC6=C(O)C(C4)=CC(S(=O)(O)=O)=C6')